# Capstone: a controlled DiffCAM versus Grad-CAM experiment

**Weeks:** 38–39 · **Core question:** Can we compare two methods under defensible experimental controls?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

Reproduce a **small synthetic evaluation** from first principles. Keep the encoder, target images, preprocessing, and evaluation metric fixed. DiffCAM uses class-0 reference embeddings; Grad-CAM uses the class-1 model score. These explain related but not identical objects, so interpretation must explicitly discuss this mismatch.

**Capstone deliverables:** an executable notebook, table of per-image metrics, three examples, and a paragraph about limits. This is NOT a replication of the paper's published accuracy or medical benchmarks.

**Learning goal:** Present a result with uncertainty, a well-defined comparison question, and explicit limitations.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from scipy.ndimage import zoom
from diffcam_edu import make_cell_images
from diffcam_edu.models import fit_cnn,feature_maps,embeddings_from_maps,gradcam
from diffcam_edu.math import diffcam_weights,diffcam_map,normalize_map
from diffcam_edu.evaluation import area_under_precision_recall
x,y,regions=make_cell_images(340,seed=24)
train=np.arange(260);test=np.arange(260,340)
model,history=fit_cnn(x[train],y[train],epochs=9,seed=4)
A=feature_maps(model,x);z=embeddings_from_maps(A)
ref_ids=np.flatnonzero(y[train]==0)
target_ids=np.array([i for i in test if y[i]==1])[:24]
print('Train/test:',len(train),len(test),'Reference samples:',len(ref_ids),'Target positive test images:',len(target_ids))

### Experiment 2 of 3

In [ ]:
records=[];examples=[]
for i in target_ids:
    weights=diffcam_weights(z[i],z[ref_ids],ridge=.1)
    diff=normalize_map(diffcam_map(A[i],weights))
    grad=normalize_map(gradcam(model,x[i],1)[0])
    factor=np.array(x[i].shape)/np.array(diff.shape)
    diff_big=zoom(diff,factor,order=1);grad_big=zoom(grad,factor,order=1)
    records.append((i,area_under_precision_recall(diff_big,regions[i]),area_under_precision_recall(grad_big,regions[i])))
    if len(examples)<3: examples.append((i,diff_big,grad_big))
import pandas as pd
scores=pd.DataFrame(records,columns=['image','DiffCAM_AUPRC','GradCAM_AUPRC'])
print(scores.describe().round(3))

### Experiment 3 of 3

In [ ]:
fig,ax=plt.subplots(3,3,figsize=(9,9))
for row,(i,diff,grad) in enumerate(examples):
    for a,img,t in zip(ax[row],[x[i],diff,grad],[f'Target {i}','DiffCAM','Grad-CAM']):
        a.imshow(img,cmap='gray' if t.startswith('Target') else 'inferno');a.set_title(t);a.axis('off')
plt.tight_layout();plt.show()
assert len(scores)>0 and np.isfinite(scores[['DiffCAM_AUPRC','GradCAM_AUPRC']].to_numpy()).all()

## Exercises — complete independently
1. Add a reference-count ablation and report uncertainty over independent test images.
2. Compare localization vs faithfulness: why is higher AUPRC not proof of better explanation of a prediction?

## Check your understanding

**Exit ticket:** Present a result with uncertainty, a well-defined comparison question, and explicit limitations.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)